# Batch Size Effects

Larger batches: **faster per epoch** but sometimes **less stable** loss. Smaller batches: noisier gradients.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Measure steps per epoch vs batch size

In [ ]:
gen = DummyDataGenerator(n_samples=512, n_features=8, n_classes=3)
ds = TabularDataset(*gen.tensors())
batch_sizes = [8, 16, 32, 64, 128]
steps_per_epoch = [len(DataLoader(ds, batch_size=bs)) for bs in batch_sizes]
print(dict(zip(batch_sizes, steps_per_epoch)))

## 2. Steps per epoch vs batch size

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(batch_sizes, steps_per_epoch, 'o-', color='steelblue', lw=2, markersize=8)
ax.set_xlabel('batch size'); ax.set_ylabel('steps per epoch')
ax.set_title('Fewer steps with larger batches (fixed dataset)')
plt.tight_layout(); plt.show()

## 3. Loss stability across batch sizes

In [ ]:
def train_trace(bs, steps=50):
    m = SimpleMLP()
    opt = torch.optim.SGD(m.parameters(), lr=0.05)
    ld = DataLoader(ds, batch_size=bs, shuffle=True)
    losses = []
    it = iter(ld)
    for _ in range(steps):
        try:
            bx, by = next(it)
        except StopIteration:
            it = iter(ld); bx, by = next(it)
        opt.zero_grad()
        l = F.cross_entropy(m(bx), by)
        l.backward(); opt.step()
        losses.append(l.item())
    return losses

traces = {bs: train_trace(bs) for bs in [8, 64]}

## 4. Compare loss traces

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(traces[8], alpha=0.8, label='batch=8 (noisy)', lw=1.5)
ax.plot(traces[64], alpha=0.8, label='batch=64 (smooth)', lw=1.5)
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()
ax.set_title('Loss stability: small vs large batch')
plt.tight_layout(); plt.show()

## 5. Simulated time per epoch vs batch size

In [ ]:
# time ≈ steps * (base + batch_factor * bs) — illustrative
time_per_epoch = [s * (0.01 + 0.0001 * bs) for s, bs in zip(steps_per_epoch, batch_sizes)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(b) for b in batch_sizes], time_per_epoch, color='coral', edgecolor='black')
ax.set_xlabel('batch size'); ax.set_ylabel('relative time / epoch')
ax.set_title('Wall-clock tradeoff (synthetic timing model)')
plt.tight_layout(); plt.show()